In [2]:

# ============================================================
# ANACCORDA - MUSICAL NOTE SYNTHESIS
# ============================================================
# This notebook generates musical notes using:
# 1. Harmonic synthesis
# 2. Slight detuning (+1 Hz)
# 3. Exponential amplitude envelope
# 4. Slow LFO modulation
# 5. Final signal mixing
#
# Each Note object contains all the audio signals generated
# during the synthesis process.
# ============================================================


# ============================================================
# 1. IMPORTS AND GLOBAL AUDIO SETTINGS
# ============================================================

# Import Audio to play generated signals inside Jupyter Notebook
# Import display to show audio players
# Import NumPy for numerical operations and signal generation


In [3]:
from IPython.display import Audio, display
import numpy as np


# Define the audio sample rate (samples per second)
SR = 48000

# Define the total duration of each generated note (seconds)
Init_duration = 3

# Calculate the total number of samples
N_global = int(SR * Init_duration)

# Create the global time axis
# endpoint=False ensures that the final sample is not duplicated
T_global = np.linspace(
    0,
    Init_duration,
    N_global,
    endpoint=False
)

# Set the initial amplitude of each harmonic stack
Amplitude_global = 0.5

In [4]:
# ============================================================
# 2. AUDIO PLAYBACK FUNCTION
# ============================================================

# Convert a NumPy waveform into an audio player
# normalize=False preserves the original signal amplitude

def play(wave, sample_rate=SR):
    return Audio(
        wave,
        rate=sample_rate,
        normalize=False
    )



In [5]:
# ============================================================
# 3. EXPONENTIAL ENVELOPE
# ============================================================

# Generate an exponential decay envelope
#
# env_duration: duration of the envelope in seconds
# tau: time constant controlling the decay speed
# sample_rate: number of samples per second
#
# A smaller tau produces a faster decay.
# A larger tau produces a slower decay.

def envelope_exp(env_duration=3, tau=0.7, sample_rate=SR):

    # Prevent invalid or zero decay constants
    if tau <= 0:
        raise ValueError("tau must be positive.")

    # Calculate the total number of envelope samples
    n_env = round(env_duration * sample_rate)

    # Generate the exponential decay
    env = [
        np.exp(-(i / sample_rate) / tau)
        for i in range(n_env)
    ]

    return env


# Generate the default envelope
env = envelope_exp()


In [6]:

# ============================================================
# 4. HARMONIC AMPLITUDES
# ============================================================

# Define the relative amplitude of each harmonic
#
# Index 0: Fundamental frequency
# Index 1: Second harmonic
# Index 2: Third harmonic
# ...
#
# These values determine the timbre of the generated note.

Amplitudes = [
    1.0,
    0.15,
    0.02,
    0.005,
    0.0025,
    0.00125
]


In [7]:

# ============================================================
# 5. PAD MIXING FUNCTION
# ============================================================

# Combine two waveforms using independent gain controls
#
# wave1: first waveform
# wave2: second waveform
# gain1: amplitude multiplier for the first waveform
# gain2: amplitude multiplier for the second waveform
#
# The second waveform will be used as a slightly detuned layer.
#
# If the resulting signal exceeds the [-1, 1] range,
# its peak amplitude is reduced to 0.9.

def pad(wave1, wave2, gain1=1, gain2=0.3):

    # Mix the two waveforms
    Pad = wave1 * gain1 + wave2 * gain2

    # Find the maximum absolute amplitude
    Peak = np.max(np.abs(Pad))

    # Reduce the signal amplitude if it exceeds the allowed range
    if Peak > 1:
        Pad = Pad / Peak * 0.9

    return Pad



In [8]:
# ============================================================
# 6. MUSICAL NOTE FREQUENCIES
# ============================================================

# Define the fundamental frequency of each note
# The dictionary covers two octaves, from C4 to C6.
#
# Each key is the note name.
# Each value is its fundamental frequency in Hz.

Notes = {

    # Fourth octave
    "Do4": 262.00,
    "Do#4": 277.58,
    "Re4": 294.33,
    "Re#4": 311.82,
    "Mi4": 330.22,
    "Fa4": 349.89,
    "Fa#4": 370.69,
    "Sol4": 392.99,
    "Sol#4": 416.36,
    "La4": 441.00,
    "La#4": 467.21,
    "Si4": 494.99,

    # Fifth octave
    "Do5": 524.00,
    "Do#5": 555.16,
    "Re5": 588.66,
    "Re#5": 623.64,
    "Mi5": 660.44,
    "Fa5": 699.78,
    "Fa#5": 741.38,
    "Sol5": 785.98,
    "Sol#5": 832.72,
    "La5": 882.00,
    "La#5": 934.42,
    "Si5": 989.98,

    # Sixth octave
    "Do6": 1048.00
}


In [9]:


# ============================================================
# 7. NOTE CLASS
# ============================================================

# The Note class generates a complete Pad sound for a given note.
#
# Each Note object contains:
# - Two harmonic stacks
# - A slightly detuned second stack (+1 Hz)
# - A mixed and enveloped signal
# - An LFO-modulated signal
# - The final Pad waveform


class Note:

    def __init__(self, note_name):

        # ----------------------------------------------------
        # 7.1 GET THE FUNDAMENTAL FREQUENCIES
        # ----------------------------------------------------

        # Retrieve the fundamental frequency from the Notes dictionary
        self.frequency1 = Notes[note_name]

        # Detune the second fundamental frequency by +1 Hz
        self.frequency2 = self.frequency1 + 1


        # ----------------------------------------------------
        # 7.2 CALCULATE THE HARMONIC FREQUENCIES
        # ----------------------------------------------------

        # Generate the first harmonic series
        # Harmonics are integer multiples of the fundamental frequency

        self.Frequencies1 = [
            self.frequency1 * (i + 1)
            for i in range(6)
        ]

        # Generate the second, slightly detuned harmonic series

        self.Frequencies2 = [
            self.frequency2 * (i + 1)
            for i in range(6)
        ]


        # ----------------------------------------------------
        # 7.3 GENERATE THE FIRST HARMONIC STACK
        # ----------------------------------------------------

        # Initialize an empty waveform
        self.Do_sinewave = np.zeros(N_global)

        # Generate each harmonic and add it to the waveform
        for freq, amp in zip(self.Frequencies1, Amplitudes):

            self.Do_sinewave += amp * np.sin(
                2 * np.pi * freq * T_global
            )

        # Apply the global amplitude to the complete harmonic stack
        self.Do_sinewave *= Amplitude_global


        # ----------------------------------------------------
        # 7.4 GENERATE THE SECOND DETUNED HARMONIC STACK
        # ----------------------------------------------------

        # Initialize the second empty waveform
        self.Do_263_sinewave = np.zeros(N_global)

        # Generate the harmonics using the detuned frequencies
        for freq, amp in zip(self.Frequencies2, Amplitudes):

            self.Do_263_sinewave += amp * np.sin(
                2 * np.pi * freq * T_global
            )

        # Apply the same global amplitude
        self.Do_263_sinewave *= Amplitude_global


        # ----------------------------------------------------
        # 7.5 MIX THE TWO HARMONIC STACKS
        # ----------------------------------------------------

        # Combine both waveforms
        # First stack gain: 1.0
        # Second stack gain: 0.3

        self.Do_pad = pad(
            self.Do_sinewave,
            self.Do_263_sinewave,
            gain1=1,
            gain2=0.3
        )


        # ----------------------------------------------------
        # 7.6 APPLY THE EXPONENTIAL ENVELOPE
        # ----------------------------------------------------

        # Multiply the mixed waveform by the exponential envelope
        # This creates the amplitude decay over time

        self.Do_pad_env = self.Do_pad * env


        # ----------------------------------------------------
        # 7.7 GENERATE THE SLOW LFO MODULATION
        # ----------------------------------------------------

        # Define the LFO frequency in Hz
        self.fm = 4

        # Generate the LFO waveform
        # 0.7: constant amplitude offset
        # 0.15: modulation depth
        # fm: modulation frequency

        self.LFO = (
            0.7
            + 0.15 * np.sin(
                2 * np.pi * self.fm * T_global
            )
        )


        # ----------------------------------------------------
        # 7.8 APPLY THE LFO MODULATION
        # ----------------------------------------------------

        # Multiply the enveloped Pad by the LFO
        # This creates the slow breathing effect

        self.Pad_mod = self.Do_pad_env * self.LFO


        # ----------------------------------------------------
        # 7.9 GENERATE THE FINAL PAD SIGNAL
        # ----------------------------------------------------

        # Mix the original enveloped signal with the modulated signal
        # Both signals have equal weight (50% each)

        self.Pad_final = (
            self.Do_pad_env + self.Pad_mod
        ) / 2



In [10]:
# ============================================================
# 8. CREATE AND TEST A NOTE
# ============================================================

# Create a C4 (Do4) Note object
Do4 = Note("Do4")

# Play the final Pad waveform
play(Do4.Pad_final)


In [11]:
Re_sharp5 = Note("Re#5")
play(Re_sharp5.Pad_final)

In [13]:
from scipy.io.wavfile import write

In [15]:
Do4 = Note("Do4")
Mi4 = Note("Mi4")
Sol4 =Note("Sol4")
Si4 = Note("Si4")

Chord = (Do4.Pad_final + Mi4.Pad_final + Sol4.Pad_final + Si4.Pad_final) / 4

play(Chord)

In [16]:
Cmaj7 = np.array(Chord, dtype=np.float32)
write("Cmaj7.wav", SR, Cmaj7)

In [17]:
for note_name in Notes:
    note = Note(note_name)
    audio = np.array(note.Pad_final, dtype=np.float32)
    write(f"{note_name}.wav", SR, audio)